## ETL
The purpose of this ETL process is to create the fact & dimentions tables.
<table style="margin-left:0">
<tr><th>Step</th><th>Purpose</th></tr>
<tr><td>Extract</td><td>read the cleaned CSV</td></tr>
<tr><td>Transform</td><td>split it into dimensions and a fact table</td></tr>
<tr><td>Load</td><td>put the results into the warehouse</td></tr>
</table>

### dim_routes

In [8]:
import pandas as pd
df = pd.read_csv("../data/cleaned/railway.csv")
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={"date_of_journey": "journey_date", "arrival_time": "scheduled_arrival_time"})

dim_routes = df[["departure_station", "arrival_destination"]].drop_duplicates().rename(columns={"arrival_destination": "arrival_station"}).reset_index(drop=True).sort_values(["departure_station", "arrival_station"])
dim_routes["route_id"] = range(1, len(dim_routes) + 1)
dim_routes = dim_routes[["route_id", "departure_station", "arrival_station"]]
dim_routes.head(3)

,route_id,departure_station,arrival_station
52,1,Birmingham New Street,Coventry
47,2,Birmingham New Street,Edinburgh Waverley
41,3,Birmingham New Street,Liverpool Lime Street


### dim_railcard

In [ ]:
dim_railcard = pd.DataFrame(sorted(df["railcard"].unique()), columns=["railcard_type"])
dim_railcard["railcard_id"] = range(1, len(dim_railcard) + 1)
dim_railcard = dim_railcard[["railcard_id", "railcard_type"]]
dim_railcard

TypeError: '<' not supported between instances of 'float' and 'str'

### dim_sale_details

In [ ]:
dim_sale_details = df[["purchase_type", "payment_method", "ticket_class", "ticket_type"]].drop_duplicates().sort_values(["purchase_type", "payment_method", "ticket_class", "ticket_type"]).reset_index(drop=True)
dim_sale_details["sale_details_id"] = range(1, len(dim_sale_details) + 1)
dim_sale_details = dim_sale_details[["sale_details_id", "purchase_type", "payment_method", "ticket_class", "ticket_type"]]
dim_sale_details.head(3)

,sale_details_id,purchase_type,payment_method,ticket_class,ticket_type
0,1,Online,Contactless,First Class,Advance
1,2,Online,Contactless,First Class,Anytime
2,3,Online,Contactless,First Class,Off-Peak


### dim_journey

In [ ]:
df = df.rename(columns={"date_of_journey": "journey_date", "arrival_time": "scheduled_arrival_time"})

journey_lookup = (
    df[["journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay", "departure_station", "arrival_destination"]]
    .drop_duplicates()
    .sort_values(["journey_date", "departure_time", "departure_station", "arrival_destination"])
    .reset_index(drop=True)
)

journey_lookup["journey_id"] = range(1, len(journey_lookup) + 1)

dim_journey = journey_lookup[["journey_id", "journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay"]]
dim_journey.head(3)

,journey_id,journey_date,departure_time,scheduled_arrival_time,actual_arrival_time,journey_status,reason_for_delay
0,1,2024-01-01,00:00:00,00:30:00,00:30:00,On Time,No Delay
1,2,2024-01-01,00:00:00,01:50:00,02:07:00,Delayed,Signal Failure
2,3,2024-01-01,01:30:00,02:30:00,02:30:00,On Time,No Delay


fact_ticket

In [ ]:
fact_ticket = (
    df.merge(dim_railcard.rename(columns={"railcard_type": "railcard"}), on="railcard", how="left", validate="m:1")
    .merge(dim_routes.rename(columns={"arrival_station": "arrival_destination"}), on=["departure_station", "arrival_destination"], how="left", validate="m:1")
    .merge(dim_sale_details, on=["purchase_type", "payment_method", "ticket_class", "ticket_type"], how="left", validate="m:1")
    .merge(journey_lookup[["journey_date", "departure_time", "departure_station", "arrival_destination", "journey_id"]], on=["journey_date", "departure_time", "departure_station", "arrival_destination"], how="left", validate="m:1")
    .rename(columns={"date_of_purchase": "purchase_date", "time_of_purchase": "purchase_time"})
    [["transaction_id", "journey_id", "railcard_id", "route_id", "sale_details_id", "purchase_date", "purchase_time", "price", "refund_request"]]
)

fact_ticket.head(3)

# fact_ticket.to_csv("../data/star/fact_ticket.csv", index=False)
# dim_railcard.to_csv("../data/star/dim_railcard.csv", index=False)
# dim_routes.to_csv("../data/star/dim_routes.csv", index=False)
# dim_sale_details.to_csv("../data/star/dim_sale_details.csv", index=False)
# dim_journey.to_csv("../data/star/dim_journey.csv", index=False)